In [29]:
import psutil
import platform
import os
env_name = os.environ.get('CONDA_DEFAULT_ENV')
print("当前 conda 环境名：", env_name)
print(platform.system()) # 操作系统名称
print(platform.release()) # 操作系统版本
print(platform.machine()) # 计算机架构
print(platform.processor()) # 处理器类型
# CPU 信息
print(psutil.cpu_count()) # CPU 核数
print(psutil.cpu_freq()) # CPU 频率
# 内存信息
print(psutil.virtual_memory()) # 内存总量、可用内存、已用内存等

当前 conda 环境名： FLLFFL
Windows
11
AMD64
Intel64 Family 6 Model 151 Stepping 2, GenuineIntel
24
scpufreq(current=2000.0, min=0.0, max=2000.0)
svmem(total=34008584192, available=11702231040, percent=65.6, used=22306353152, free=11702231040)


## 2.1


In [16]:
import math

# 输入参数
C_in = 3              # 输入通道数
H_in, W_in = 32, 32   # 输入空间尺寸
K = 5                 # 卷积核空间尺寸 (5×5)
P = 2                 # Padding
S = 2                 # Stride
num_filters = 16      # 卷积核数量，即输出通道数

# 输出尺寸公式: H_out = floor((H_in + 2P - K) / S) + 1
H_out = math.floor((H_in + 2 * P - K) / S) + 1
W_out = math.floor((W_in + 2 * P - K) / S) + 1
C_out = num_filters

print(f"输出特征图尺寸: {C_out} × {H_out} × {W_out}")

输出特征图尺寸: 16 × 16 × 16


In [17]:
# 单个输出像素的乘法次数 = 输入通道数 × 卷积核高 × 卷积核宽
# 每个输出像素由一个卷积核在所有输入通道的感受野上逐元素相乘再求和得到
multiplications = C_in * K * K  # 3 × 5 × 5 = 75
print(multiplications)

75


## 2.2



In [18]:
import numpy as np

def max_pool2d_forward(x, kernel_size, stride=2, padding=0):
    # 将 int 类型的参数统一转为 (h, w) 元组
    if isinstance(kernel_size, int):
        kernel_size = (kernel_size, kernel_size)
    if isinstance(stride, int):
        stride = (stride, stride)
    if isinstance(padding, int):
        padding = (padding, padding)

    kh, kw = kernel_size  # 池化窗口的高和宽
    sh, sw = stride      # 步幅的高和宽
    ph, pw = padding     # 填充的高和宽

    # 统一升维到 4D (N, C, H, W)，方便统一处理
    original_ndim = x.ndim
    if original_ndim == 2:
        x = x[np.newaxis, np.newaxis, :, :]  # (H,W) -> (1,1,H,W)
    elif original_ndim == 3:
        x = x[np.newaxis, :, :, :]            # (C,H,W) -> (1,C,H,W)

    N, C, H, W = x.shape

    # 对输入做零填充，用 -inf 填充使 padding 区域不影响 max 计算
    if ph > 0 or pw > 0:
        x_padded = np.pad(x, ((0, 0), (0, 0), (ph, ph), (pw, pw)),
                          mode='constant', constant_values=-np.inf)
    else:
        x_padded = x

    H_pad, W_pad = x_padded.shape[2], x_padded.shape[3]

    # 计算输出空间尺寸
    H_out = (H_pad - kh) // sh + 1
    W_out = (W_pad - kw) // sw + 1

    out = np.zeros((N, C, H_out, W_out))

    # 滑动窗口：在每个位置取窗口内的最大值
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * sh
            w_start = j * sw
            window = x_padded[:, :, h_start:h_start + kh, w_start:w_start + kw]
            out[:, :, i, j] = np.max(window, axis=(2, 3))  # 在空间维度上取 max

    # 恢复与输入相同的维度
    if original_ndim == 2:
        out = out[0, 0, :, :]
    elif original_ndim == 3:
        out = out[0, :, :, :]

    return out


# 测试：输入 (1,1,6,6)，分别用两组不同参数
np.random.seed(42)
x = np.random.randn(1, 1, 6, 6)
out1 = max_pool2d_forward(x, kernel_size=2, stride=2, padding=0)  # 输出 3×3
out2 = max_pool2d_forward(x, kernel_size=3, stride=1, padding=1)  # 输出 6×6
print(out1.shape, out2.shape)

(1, 1, 3, 3) (1, 1, 6, 6)


## 3.1


In [19]:
C = 64

# 单层 5×5 卷积参数量 = C_in × C_out × 5 × 5（此处 C_in = C_out = C）
params_5x5 = C * C * 5 * 5  # 64×64×25 = 102400
print(params_5x5)

102400


In [20]:
# 两层 3×3 卷积参数量 = 2 × (C × C × 3 × 3)
# 两层 3×3 的感受野等价于一层 5×5，但参数更少
params_two_3x3 = 2 * C * C * 3 * 3  # 2×64×64×9 = 73728
print(params_two_3x3)

73728


## 3.2



In [21]:
import torch.nn as nn

def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    """NiN 块：一个空间卷积 + 两个 1×1 卷积（起到逐像素全连接的作用）"""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),  # 空间卷积提取特征
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),  # 1×1 卷积：通道间融合
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),  # 1×1 卷积：增加非线性
        nn.ReLU()
    )

# 示例：输入 192 通道，输出 256 通道，5×5 卷积 + padding=2 保持尺寸
block = nin_block(192, 256, kernel_size=5, stride=1, padding=2)
print(block)

Sequential(
  (0): Conv2d(192, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (1): ReLU()
  (2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU()
  (4): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU()
)


## 4.1



In [22]:
import numpy as np

x = np.array([2.0, 4.0, 6.0, 8.0])
gamma = 2.0   # 缩放参数
beta = 1.0    # 平移参数
eps = 0.0     # 防止除零的小常数

# BN 前向传播步骤
mu = np.mean(x)                    # 1. 计算均值
var = np.var(x)                    # 2. 计算方差
x_hat = (x - mu) / np.sqrt(var + eps)  # 3. 标准化
y = gamma * x_hat + beta           # 4. 缩放 + 平移
print(y)

[-1.68328157  0.10557281  1.89442719  3.68328157]


## 4.2



In [23]:
import torch
from torch import nn
from torch.nn import functional as F

class Residual(nn.Module):
    def __init__(self, input_channels, num_channels, use_1x1conv=False, strides=1):
        """
        残差块：两层 3×3 卷积 + BN + ReLU，可选 1×1 卷积对齐通道/尺寸
        use_1x1conv: 是否用 1×1 卷积将残差映射到与主路径相同的形状
        strides: 第一个卷积的步幅，可下采样
        """
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, num_channels, kernel_size=3, padding=1, stride=strides)
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.conv2 = nn.Conv2d(num_channels, num_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_channels)
        # 1×1 卷积：当输入输出通道不同或需要下采样时使用
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels, num_channels, kernel_size=1, stride=strides)
        else:
            self.conv3 = None

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))  # 第一个卷积 + BN + ReLU
        Y = self.bn2(self.conv2(Y))           # 第二个卷积 + BN（暂不加 ReLU）
        if self.conv3:
            X = self.conv3(X)  # 对捷径做 1×1 卷积以匹配形状
        Y += X                  # 残差连接：F(X) + X
        return F.relu(Y)        # 合并后再过 ReLU


# 测试
blk1 = Residual(3, 3)                               # 通道不变，无需 1×1 卷积
blk2 = Residual(3, 6, use_1x1conv=True, strides=2)  # 通道变化 + 下采样
X = torch.randn(4, 3, 6, 6)
print(blk1(X).shape)  # 期望 (4, 3, 6, 6)
print(blk2(X).shape)  # 期望 (4, 6, 3, 3)

torch.Size([4, 3, 6, 6])
torch.Size([4, 6, 3, 3])


## 5.1



In [24]:
print('''1. 底层特征提取层在大型源数据集上预训练后，已经学习到了通用的、鲁棒的低级特征（如边缘、纹理、形状等），这些特征在新任务中通常仍然有效。
设置较小学习率或冻结参数，可以保留这些已学到的通用知识，避免在少量目标数据上被快速破坏。
顶层输出层（分类器）是新初始化的，需要针对新数据集的类别分布进行从头学习，因此需要较大学习率以快速收敛。''')

1. 底层特征提取层在大型源数据集上预训练后，已经学习到了通用的、鲁棒的低级特征（如边缘、纹理、形状等），这些特征在新任务中通常仍然有效。
设置较小学习率或冻结参数，可以保留这些已学到的通用知识，避免在少量目标数据上被快速破坏。
顶层输出层（分类器）是新初始化的，需要针对新数据集的类别分布进行从头学习，因此需要较大学习率以快速收敛。


In [25]:
print('''2. 采取的策略：
- 冻结底层特征提取层的全部参数，只训练顶层输出层（或最后几层），防止在小数据集上过拟合。
- 使用较小的学习率和较早的早停（Early Stopping）。
- 增加正则化手段，如权重衰减（Weight Decay）和 Dropout。
- 进行图像增广以扩充有效训练样本。''')

2. 采取的策略：
- 冻结底层特征提取层的全部参数，只训练顶层输出层（或最后几层），防止在小数据集上过拟合。
- 使用较小的学习率和较早的早停（Early Stopping）。
- 增加正则化手段，如权重衰减（Weight Decay）和 Dropout。
- 进行图像增广以扩充有效训练样本。


## 5.2



In [26]:
from torchvision import transforms

# 图像增广管道：随机裁剪 + 随机翻转 + 颜色抖动 + 转 Tensor
augmentation_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),  # 随机缩放裁剪到 224×224
    transforms.RandomHorizontalFlip(p=0.5),                # 50% 概率水平翻转
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),  # 随机颜色抖动
    transforms.ToTensor()  # 转为 [0,1] 的 Tensor
])

print(augmentation_pipeline)

Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)


## 6.1



In [27]:
# 边界框格式: [x1, y1, x2, y2]（左上角 + 右下角）
A = [10, 10, 50, 50]
B = [30, 30, 70, 70]

# 1. 计算交集区域的坐标
inter_x1 = max(A[0], B[0])
inter_y1 = max(A[1], B[1])
inter_x2 = min(A[2], B[2])
inter_y2 = min(A[3], B[3])

# 2. 计算交集面积（若不相交则宽或高为 0）
inter_w = max(0, inter_x2 - inter_x1)
inter_h = max(0, inter_y2 - inter_y1)
inter_area = inter_w * inter_h

# 3. 计算各自面积
area_A = (A[2] - A[0]) * (A[3] - A[1])
area_B = (B[2] - B[0]) * (B[3] - B[1])

# 4. 并集面积 = A + B - 交集
union_area = area_A + area_B - inter_area

# 5. IoU = 交集 / 并集
iou = inter_area / union_area
print(iou)

0.14285714285714285


## 6.2



In [28]:
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, targets, epsilon=0.1):
    """
    标签平滑交叉熵：将 one-hot 标签的少量概率分配给非目标类，
    防止模型过于自信，提升泛化能力。
    L = (1-ε)*L_CE + ε*L_smooth
    """
    K = logits.size(-1)                      # 类别数
    log_probs = F.log_softmax(logits, dim=-1) # log softmax
    nll_loss = F.nll_loss(log_probs, targets)  # 标准交叉熵部分
    smooth_loss = -log_probs.mean(dim=-1).mean()  # 均匀分布部分
    loss = (1 - epsilon) * nll_loss + epsilon * smooth_loss  # 加权组合
    return loss


# 测试：4 个样本，10 个类别
logits = torch.randn(4, 10)
targets = torch.tensor([1, 3, 5, 7])
loss = label_smoothing_cross_entropy(logits, targets, epsilon=0.1)
print(loss.item())

2.4753100872039795
